In [29]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [30]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/NGC-3049_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/NGC-3049_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     387   (2094, 965)   float32   
  1  IMAGE.ERR     1 ImageHDU        68   (2094, 965)   float32   


In [31]:
image_cut = image[67:167, 0:1890]
image_error_cut = image_error[67:167, 0:1890]

In [32]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00485
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [33]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) #+
       # sersic_1d(x_hr, params4)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [34]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/PSF/Real_seeing_NGC3049.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [35]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, a, b):
    func = model_convolved(x, params1, params2, params3)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    sum_area = blue_area + green_area +purple_area 
    total_area = total_integral(x, params1, params2, params3, a, b)
    print(blue_area)
    print(green_area)
    print(purple_area)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area]     

## Halpha

In [36]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/Fit/Halpha_fit.csv", index_col=0)

In [37]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 1'].iloc[3] -1, 
                                                                fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 2'].iloc[3] -1, 
                                                                fit_HA['Component 2'].iloc[3] + 1)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 3'].iloc[3] -1, 
                                                                fit_HA['Component 3'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

229.81558669815132
0.3038025978096549
0.02425143810358439
230.14364073406458
157.20758858067026
11.546665833270254
15.42648379253443
0.12789073947561005
27.101040365280294
24.898882816948856
0.6532074977368665
0.002912253381803475
22.287187603711608
22.943307354830278
22.41530196951901
           Peak 1     Peak 2     Peak 3
Blue    99.857457  42.605987   2.847050
Green    0.132006  56.922109   0.012693
Purple   0.010538   0.471903  97.140256


In [38]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

229.81558669815132
0.3038025978096549
0.02425143810358439
230.14364073406458
157.20758858067026
11.546665833270254
15.42648379253443
0.12789073947561005
27.101040365280294
24.898882816948856
0.6532074977368665
0.002912253381803475
22.287187603711608
22.943307354830278
22.41530196951901
           Peak 1     Peak 2     Peak 3
Blue    99.857457  42.605987   2.847050
Green    0.132006  56.922109   0.012693
Purple   0.010538   0.471903  97.140256


In [39]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

349.9631723255743
0.634396988347853
0.04905027517138982
350.6466195890936
262.58286056924317
23.42578574760533
24.759408173718708
0.2588821324259754
48.44407605375002
45.83957490507943
1.3099600875339992
0.005949792395215674
41.0082739619
42.32418384182922
41.71061491176953
           Peak 1     Peak 2     Peak 3
Blue    99.805089  48.356347   3.095063
Green    0.180922  51.109259   0.014058
Purple   0.013989   0.534394  96.890879


## HBeta


In [40]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/Fit/Hbeta_fit.csv", index_col = 0)

In [41]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -1, 
                                                                fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -1, 
                                                                fit_HA['Component 2'].iloc[3] + 1)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -1, 
                                                                fit_HA['Component 3'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

565.0007548315077
2.0555315660300932e-11
0.17930257831523644
565.1800574098436
100.34313222220216
3.922668240969365
6.953825480352952
0.2812221044694003
11.157715825791717
11.186082132652697
0.3913127387471501
3.3554777383758974e-249
9.05806011322064
9.449372851967789
10.183003023604027
              Peak 1     Peak 2         Peak 3
Blue    9.996828e+01  35.156553   4.141150e+00
Green   3.636950e-12  62.323020  3.551006e-248
Purple  3.172486e-02   2.520427   9.585885e+01


In [42]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

565.0007548315077
2.0555315660300932e-11
0.17930257831523644
565.1800574098436
100.34313222220216
3.922668240969365
6.953825480352952
0.2812221044694003
11.157715825791717
11.186082132652697
0.3913127387471501
3.3554777383758974e-249
9.05806011322064
9.449372851967789
10.183003023604027
              Peak 1     Peak 2         Peak 3
Blue    9.996828e+01  35.156553   4.141150e+00
Green   3.636950e-12  62.323020  3.551006e-248
Purple  3.172486e-02   2.520427   9.585885e+01


In [43]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


718.0807746208254
7.680735328317871e-09
0.3589618360636967
718.4397364645698
152.33125416322133
7.9533591719239745
12.025070418545212
0.5633354715656376
20.541765062034823
20.579796975190447
0.7838615213105831
3.125848549839857e-227
16.57712430138308
17.360985822693664
18.268813696296917
              Peak 1     Peak 2         Peak 3
Blue    9.995004e+01  38.717993   4.515075e+00
Green   1.069086e-09  58.539616  1.800502e-226
Purple  4.996408e-02   2.742391   9.548493e+01


## NII

In [44]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/Fit/NII_fit.csv", index_col = 0)

In [45]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] - 1, 
                                                                fit_HA['Component 1'].iloc[3] +  1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] - 1, 
                                                                fit_HA['Component 2'].iloc[3] +  1)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] - 1, 
                                                                fit_HA['Component 3'].iloc[3] +  1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

91.57529208260082
0.1909147320730722
0.22584089475642838
91.99204770943032
65.42340503106217
5.069082558197827
8.608679113099065
0.36374105690785435
14.041502728204746
14.235217421130564
0.4693104618235207
0.0023539781503104425
8.523950721672024
8.995615161645855
9.790345154885044
           Peak 1     Peak 2     Peak 3
Blue    99.546966  36.100713   5.217102
Green    0.207534  61.308816   0.026168
Purple   0.245500   2.590471  94.756729


In [46]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

91.57529208260082
0.1909147320730722
0.22584089475642838
91.99204770943032
65.42340503106217
5.069082558197827
8.608679113099065
0.36374105690785435
14.041502728204746
14.235217421130564
0.4693104618235207
0.0023539781503104425
8.523950721672024
8.995615161645855
9.790345154885044
           Peak 1     Peak 2     Peak 3
Blue    99.546966  36.100713   5.217102
Green    0.207534  61.308816   0.026168
Purple   0.245500   2.590471  94.756729


In [47]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

138.86193156048904
0.39810220952821507
0.45216999603949437
139.71220376605675
107.72525632863653
10.260067661150362
14.233212246879253
0.7286834395499152
25.22196334757953
25.52449641127201
0.9403229749838468
0.004789651506651382
15.416061312003656
16.361173938494154
17.4408916493101
           Peak 1     Peak 2     Peak 3
Blue    99.391412  40.679100   5.747283
Green    0.284944  56.431817   0.029274
Purple   0.323644   2.889083  94.223442


## SII

In [48]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/Fit/SII_fit.csv", index_col = 0)

In [49]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -1, 
                                                                fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -1, 
                                                                fit_HA['Component 2'].iloc[3] + 1)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -1, 
                                                                fit_HA['Component 3'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

19.366841352525228
0.14077723697486383
0.14762455502660188
19.655243144526693
19.294034620710512
3.3465842520354343
6.531971369487341
0.23525616575057118
10.113811787273345
10.096975550749281
0.5319303794873711
0.0017203169057818798
4.349598867908965
4.883249564302118
5.1926484028712965
           Peak 1     Peak 2     Peak 3
Blue    98.532698  33.089248  10.892959
Green    0.716232  64.584664   0.035229
Purple   0.751070   2.326088  89.071812


In [50]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

19.366841352525228
0.14077723697486383
0.14762455502660188
19.655243144526693
19.294034620710512
3.3465842520354343
6.531971369487341
0.23525616575057118
10.113811787273345
10.096975550749281
0.5319303794873711
0.0017203169057818798
4.349598867908965
4.883249564302118
5.1926484028712965
           Peak 1     Peak 2     Peak 3
Blue    98.532698  33.089248  10.892959
Green    0.716232  64.584664   0.035229
Purple   0.751070   2.326088  89.071812


In [51]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

33.71585997502485
0.2935592033813268
0.2955542166194117
34.30497339502559
33.9579252759102
6.741454160513833
10.726091665004384
0.47124186802230517
17.938787693540522
17.96285260719376
1.0651277382482365
0.0035014624713324183
7.9623996957740175
9.031028896493586
9.450297328087595
           Peak 1     Peak 2     Peak 3
Blue    98.282717  37.580322  11.794091
Green    0.855734  59.792734   0.038771
Purple   0.861549   2.626944  88.167138


## OIII

In [52]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/Fit/OIII_fit.csv", index_col = 0)

In [53]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -1, 
                                                                fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -1, 
                                                                fit_HA['Component 2'].iloc[3] + 1)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -1, 
                                                                fit_HA['Component 3'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

56.93293111904126
0.0010831292053760148
0.09304836974548575
57.02706261799212
39.62287823933763
2.6625776694425114
8.498918899563947
0.15081841256873446
11.312314981575192
11.341706298208564
0.29436392483808016
2.318527943477622e-17
6.501924187731753
6.796288112569833
7.668158998638644
           Peak 1     Peak 2        Peak 3
Blue    99.834935  23.536983  4.331246e+00
Green    0.001899  75.129794  3.411462e-16
Purple   0.163165   1.333223  9.566875e+01


In [54]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

56.93293111904126
0.0010831292053760148
0.09304836974548575
57.02706261799212
39.62287823933763
2.6625776694425114
8.498918899563947
0.15081841256873446
11.312314981575192
11.341706298208564
0.29436392483808016
2.318527943477622e-17
6.501924187731753
6.796288112569833
7.668158998638644
           Peak 1     Peak 2        Peak 3
Blue    99.834935  23.536983  4.331246e+00
Green    0.001899  75.129794  3.411462e-16
Purple   0.163165   1.333223  9.566875e+01


In [55]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

84.191964163298
0.0038601687009540177
0.18630543967051358
84.38212977166947
63.43713198693298
5.387418721220492
14.484217581987114
0.3021767588393719
20.173813062046978
20.224726370033345
0.5896202087622141
1.4227168342084077e-16
11.818996529826572
12.408616738588787
13.585648832068316
           Peak 1     Peak 2        Peak 3
Blue    99.774638  26.705010  4.751700e+00
Green    0.004575  71.797124  1.146556e-15
Purple   0.220788   1.497866  9.524830e+01
